# MobileNetV3 Large — Face Recognition Training (Colab)

**Mục tiêu:** Train MobileNetV3 cho face recognition (single-task, không KD).

| Thành phần | Chi tiết |
|---|---|
| Backbone | MobileNetV3 Large (pretrained ImageNet) |
| Embedding | 512-D (BN → AdaptiveAvgPool → Flatten) |
| Loss | MagFace (class-balanced, adaptive margin) |
| Sampler | PK Sampler — mỗi batch có `batch_size` identity khác nhau |
| Metric | AUC cosine similarity + AUC euclidean distance |

## 1. Setup môi trường

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_URL    = 'https://github.com/NguyenXuanBinh22/DATN.git'
REPO_BRANCH = 'convnext-v2-dev'
REPO_DIR    = '/content/FR_Photometric_Stereo'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin {REPO_BRANCH}')
    print('Repo đã tồn tại, đã pull latest.')

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

os.system('pip install -q albumentations==1.3.1 timm tabulate termcolor')

## 2. Imports & Cấu hình

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
import pandas as pd
import albumentations as A
from tabulate import tabulate
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from going_modular.dataloader.multitask import create_multitask_datafetcher, create_eval_loaders
from going_modular.model.FaceRecognitionMobileNetV3 import FaceRecognitionMobileNetV3
from going_modular.loss.FaceRecognitionLoss import FaceRecognitionLoss
from going_modular.train_eval.fr_train import fit
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise
from going_modular.utils.roc_auc_id import (
    compute_id_auc, compute_rank1,
    compute_id_auc_gallery_probe, compute_rank1_gallery_probe,
)
from going_modular.utils.MultiMetricEarlyStopping import MultiMetricEarlyStopping
from going_modular.utils.ModelCheckPoint import ModelCheckpoint
from going_modular.utils.ExperimentManager import ExperimentManager

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ════════════════════════════════════════════════════════════
#  CẤU HÌNH — chỉnh sửa ở đây
# ════════════════════════════════════════════════════════════

EXPERIMENT_NAME = 'MobileNetV3_Albedo_PK'

# Tên file CSV train/test (thay đổi nếu dùng split khác)
FILE_TRAIN   = 'train_split.csv'
FILE_TEST    = 'test_split.csv'

# Tên file CSV cho đánh giá gallery-probe
FILE_GALLERY = 'gallery_split.csv'
FILE_PROBE   = 'probe_split.csv'

CONFIGURATION = {
    'note':        EXPERIMENT_NAME,

    # Đường dẫn dataset trên Google Drive
    'dataset_dir': '/content/drive/MyDrive/Photometric_DB_Full/',
    'output_dir':  '/content/drive/MyDrive/mobilenetV3',
    # Modality: 'albedo' | 'normalmap' | 'depthmap'
    'type':        'albedo',

    # Backbone timm name
    'backbone':    'mobilenetv3_large_100',

    # PK Sampler: True = mỗi batch gồm batch_size identity khác nhau
    'use_sampler': True,

    'device':      device,
    'epochs':      5,
    'batch_size':  32,
    'image_size':  112,
    'base_lr':     1e-4,
    'num_classes': None,   # tự động lấy từ CSV
}

print(f"Dataset dir : {CONFIGURATION['dataset_dir']}")
print(f"File train  : {FILE_TRAIN}")
print(f"File test   : {FILE_TEST}")
print(f"File gallery: {FILE_GALLERY}")
print(f"File probe  : {FILE_PROBE}")

## 3. Data Loading

In [ ]:
import os

dataset_dir = CONFIGURATION['dataset_dir']
train_csv   = os.path.join(dataset_dir, FILE_TRAIN)
test_csv    = os.path.join(dataset_dir, FILE_TEST)

if not os.path.exists(train_csv):
    raise FileNotFoundError(f'Không tìm thấy {FILE_TRAIN} tại {dataset_dir}')

df_train = pd.read_csv(train_csv)
df_test  = pd.read_csv(test_csv)
CONFIGURATION['num_classes'] = int(df_train['id'].nunique())

print(f'num_classes      : {CONFIGURATION["num_classes"]}')
print(f'Số mẫu train     : {len(df_train)}')
print(f'Số mẫu test      : {len(df_test)}')

train_transform = A.Compose([
    RandomResizedCropRect(CONFIGURATION['image_size']),
    GaussianNoise(p=0.2),
    A.HorizontalFlip(p=0.5),
])
test_transform = A.Compose([
    A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size']),
])

train_dl, test_dl, _ = create_multitask_datafetcher(
    CONFIGURATION, train_transform, test_transform,
    file_train=FILE_TRAIN, file_test=FILE_TEST,
)
print(f'Train batches: {len(train_dl)} | Test batches: {len(test_dl)}')

# Gallery-probe loaders — CHỈ dùng cho đánh giá cuối, không dùng trong training
# gallery_split.csv: reference set | probe_split.csv: query set
gallery_dl, probe_dl = create_eval_loaders(
    CONFIGURATION, test_transform,
    gallery_csv_name=FILE_GALLERY,
    probe_csv_name=FILE_PROBE,
)

## 4. Model, Loss, Optimizer

In [ ]:
model = FaceRecognitionMobileNetV3(
    num_classes=CONFIGURATION['num_classes'],
    backbone=CONFIGURATION['backbone'],
)
model.to(device)

criterion = FaceRecognitionLoss(metadata_path=train_csv)

optimizer = Adam(model.parameters(), lr=CONFIGURATION['base_lr'])
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

# Smoke test
with torch.no_grad():
    dummy = torch.randn(2, 3, 112, 112).to(device)
    emb = model.get_embedding(dummy)
    print(f'Embedding shape : {emb.shape}')   # expect [2, 512]

## 5. Training

In [ ]:
manager = ExperimentManager(CONFIGURATION)
manager.log_text(
    f"Backbone: {CONFIGURATION['backbone']} | "
    f"Modality: {CONFIGURATION['type']} | "
    f"PK Sampler: {CONFIGURATION['use_sampler']}"
)
print(f'Checkpoint dir: {manager.ckpt_dir}')

ckpt_saver = ModelCheckpoint(
    output_dir=manager.ckpt_dir,
    mode='max',
    best_metric_name='auc_id_cosine',
)

early_stopping = MultiMetricEarlyStopping(
    monitor_keys=['auc_id_cosine'],
    patience=20,
    mode='max',
    verbose=1,
    save_dir=manager.ckpt_dir,
    start_from_epoch=5,
)

In [ ]:
fit(
    conf=CONFIGURATION,
    start_epoch=0,
    model=model,
    train_dataloader=train_dl,
    test_dataloader=test_dl,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    early_stopping=early_stopping,
    model_checkpoint=ckpt_saver,
    existing_manager=manager,
)

## 6. Resume Training từ Checkpoint

> Chạy cell này khi muốn tiếp tục train từ lần dừng trước.  
> Trỏ `CKPT_PATH` vào file checkpoint đã lưu trên Drive.

In [ ]:
CKPT_PATH = os.path.join(manager.ckpt_dir, 'last_model.pth')  # hoặc 'best_model.pth'

checkpoint = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
if 'scheduler_state_dict' in checkpoint:
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

start_epoch = checkpoint['epoch']
print(f'Resume từ epoch {start_epoch}')

fit(
    conf=CONFIGURATION,
    start_epoch=start_epoch,
    model=model,
    train_dataloader=train_dl,
    test_dataloader=test_dl,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    early_stopping=early_stopping,
    model_checkpoint=ckpt_saver,
    existing_manager=manager,
)

## 7. Đánh giá Metric (Final)

Load best checkpoint và tính AUC trên cả train lẫn test.

In [ ]:
best_ckpt = torch.load(os.path.join(manager.ckpt_dir, 'best_model.pth'), map_location=device, weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()
print(f"Best model từ epoch {best_ckpt.get('epoch', '?')}")

gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, model, device)
gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, model, device)

rows = [
    ['Cosine AUC    (gallery→probe)', f"{gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)', f"{gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)', f"{gp_rank1:.4f}"],
]
print(f"\nModel: {CONFIGURATION['backbone']} | Modality: {CONFIGURATION['type']}")
print(tabulate(rows, headers=['Metric', 'Value'], tablefmt='fancy_grid'))

## 8. Export ONNX

Export backbone + embedding (bỏ MagLinear) để chuẩn bị quantize INT8.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class InferenceModel(nn.Module):
    """Backbone + embedding (không có MagLinear) — dùng cho inference/quantize."""
    def __init__(self, fr_model):
        super().__init__()
        self.backbone  = fr_model.backbone
        self.embedding = fr_model.embedding

    def forward(self, x):
        emb = self.embedding(self.backbone(x))
        return F.normalize(emb, p=2, dim=1)

inference_model = InferenceModel(model).eval().to('cpu')
dummy_input = torch.randn(1, 3, 112, 112)
onnx_path   = os.path.join(manager.ckpt_dir, 'mobilenetv3_fr.onnx')

torch.onnx.export(
    inference_model,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17,
)
print(f'Exported ONNX: {onnx_path}')

## 9. Lưu Checkpoint lên Google Drive

Copy checkpoint ra Drive để không mất khi Colab session hết.

In [ ]:
import shutil

DRIVE_SAVE_DIR = f'/content/drive/MyDrive/checkpoints/{EXPERIMENT_NAME}/'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

for fname in ('best_model.pth', 'last_model.pth', 'mobilenetv3_fr.onnx'):
    src = os.path.join(manager.ckpt_dir, fname)
    dst = os.path.join(DRIVE_SAVE_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved: {dst}')
    else:
        print(f'Không tìm thấy: {src}')